# ASIC1a pI-code MD, part 1 of 2: prepare the CHARMM-GUI inputs

Taner Karagol and Alper Karagol.
Structure and discipline follow `MD_CHARMMGUI_GROMACS_template_2026.ipynb`.

**Runtime > Run all, then close it.** No GPU needed and nothing to fill in. It takes
under a minute.

It downloads the deposited structure, trims it to the extracellular trimer, writes the
three builds you will upload to CHARMM-GUI, and writes the settings checklist beside
them. Part 2 picks up from the CHARMM-GUI output and needs no further preparation.

## The question

The functional half of this project is finished and the answer is negative. Holding
position, charge change and side-chain volume fixed across 101 substitutions, pI-code
ladder membership changes the measured activation shift by -0.006 pH units
(permutation p = 0.895). Charge change carries the whole effect at +0.162 pH units per
charge unit.

That is a statement about a channel midpoint. It is not a statement about structure,
and the ladder's premise is structural: a near-isosteric swap is supposed to disturb
the fold less. These two notebooks put that premise to a direct test.

## Why position 246

Of the 35 charge-matched pairs in the measured set, K246 is the sharpest case against
the method, and it is not a single comparison:

| at K246 | measured shift | |
|---|---|---|
| K246Q, **on the ladder** | 0.420 | the near-isosteric move |
| K246M, off the ladder | 0.190 | ladder loses by 0.230 |
| K246A, off the ladder | 0.170 | ladder loses by 0.250 |
| K246V, off the ladder | 0.310 | ladder loses by 0.110 |

All three comparators carry the same charge change from lysine, and all three agree.
K246 is also the most buried scanned position in the protein at 1.4 percent relative
side-chain accessibility, and the closest to the acidic pocket centre at 3.7 angstrom.
Burial is the one descriptor that does predict the measured shift (rho -0.595 over 38
positions, p < 0.0001), so the position where the ladder loses hardest is also the one
where the environment should matter most.

If isostericity bought tolerance, the glutamine build should sit closer to wild type
than the methionine build. If the measurement is right, it will not.

## Why the whole trimer

In the deposited low-pH trimer 9E4F the NZ of K246 in chain A sits

    6.93 angstrom from D409 of chain B
    10.04 angstrom from E219 of chain B
    11.53 angstrom from D237 of its own chain A
    11.78 angstrom from D351 of its own chain A

K246 reaches across the subunit interface into the neighbour's pocket. A monomer
simulation would answer a different question, and both notebooks refuse a system that
is not a trimer.

## Scope, stated up front

The extracellular domain only, residues 67 to 427 of all three chains, solvated, with
no membrane and no transmembrane helices. UniProt P78348 gives TRANSMEM 50..66 and
428..458, so the cut is clean and the pocket is 25 angstrom or more from either
boundary. This buys a system of roughly 110,000 atoms instead of 200,000 and makes
three builds affordable. It costs any coupling between the pocket and the gate, which
these notebooks cannot see and do not claim to.

## What you do between the two notebooks

CHARMM-GUI is a web upload and cannot be driven from a notebook, which is why this is
split in two. Run Solution Builder three times, once per PDB this notebook writes, with
the settings it prints. Put each `gromacs/` folder on Drive where it tells you. Then
open the three part 2 notebooks, one per build, and press Run all on each.

In [ ]:
#@title Run all starts here: mount Drive, build the three inputs
PDB_ID        = "9E4F"   #@param {type:"string"}
#@markdown 9E4F is the low-pH human structure at pH 5.7. 9E4A is the resting one at 8.5.
ECD_FIRST     = 67       #@param {type:"integer"}
ECD_LAST      = 427      #@param {type:"integer"}
#@markdown UniProt P78348 gives TRANSMEM 50..66 and 428..458, so the
#@markdown extracellular domain lies between them.
POSITION      = 246      #@param {type:"integer"}
WT_RESNAME    = "LYS"    #@param {type:"string"}
LADDER_TO     = "GLN"    #@param ["GLN", "ASN", "ASP", "GLU", "HSD", "HSE", "LYS", "ARG"]
COMPARE_TO    = "MET"    #@param {type:"string"}
PROJECT       = "/content/drive/MyDrive/MDs/ASIC1a_picode"  #@param {type:"string"}
#@markdown For E418 instead: POSITION 418, WT_RESNAME GLU, LADDER_TO GLN,
#@markdown COMPARE_TO ALA, and a different PROJECT folder.

from google.colab import drive
drive.mount('/content/drive')

import os, sys, math, json, urllib.request, collections

OUT_DIR = os.path.join(PROJECT, "inputs")

# Side chains outward from CB; a substitution keeps what both share.
SIDECHAIN = {
    "LYS": ["CB", "CG", "CD", "CE", "NZ"],
    "GLN": ["CB", "CG", "CD", "OE1", "NE2"],
    "ASN": ["CB", "CG", "OD1", "ND2"],
    "GLU": ["CB", "CG", "CD", "OE1", "OE2"],
    "ASP": ["CB", "CG", "OD1", "OD2"],
    "MET": ["CB", "CG", "SD", "CE"],
    "ALA": ["CB"],
    "VAL": ["CB", "CG1", "CG2"],
    "ARG": ["CB", "CG", "CD", "NE", "CZ", "NH1", "NH2"],
    "HSD": ["CB", "CG", "ND1", "CD2", "CE1", "NE2"],
    "HSE": ["CB", "CG", "ND1", "CD2", "CE1", "NE2"],
}
BACKBONE = ["N", "CA", "C", "O"]
EXPECT_SS = 7          # extracellular disulfides per chain in human ASIC1a


def fatal(msg):
    raise SystemExit("STOP: " + msg)


def fetch(pdb_id):
    """The deposited biological assembly, not the asymmetric unit."""
    os.makedirs(OUT_DIR, exist_ok=True)
    local = os.path.join(OUT_DIR, "%s.pdb1" % pdb_id)
    if os.path.isfile(local):
        print("using the copy already in", OUT_DIR)
        return open(local, errors="replace").read()
    url = "https://files.rcsb.org/download/%s.pdb1" % pdb_id
    print("downloading", url)
    try:
        txt = urllib.request.urlopen(url, timeout=120).read().decode("utf-8", "replace")
    except Exception as e:
        fatal("could not download %s (%s). Put the file at %s by hand and rerun."
              % (url, e, local))
    open(local, "w").write(txt)
    return txt


raw = fetch(PDB_ID)
lines = []
for L in raw.splitlines():
    if not L.startswith(("ATOM", "HETATM")):
        continue
    if L[17:20] == "HOH" or L[16] not in (" ", "A"):
        continue
    lines.append(L)
if not lines:
    fatal("%s carries no usable ATOM records" % PDB_ID)

chains = sorted({L[21] for L in lines})
if len(chains) != 3:
    fatal("expected a trimer, found %d chains (%s). The acidic pocket is completed "
          "across the subunit interface, so a monomer answers a different question."
          % (len(chains), ",".join(chains)))

res = collections.OrderedDict()
for L in lines:
    res.setdefault((L[21], int(L[22:26])), L[17:20].strip())
for c in chains:
    rs = sorted(r for ch, r in res if ch == c)
    gaps = [(rs[i], rs[i + 1]) for i in range(len(rs) - 1) if rs[i + 1] - rs[i] > 1]
    inside = [r for r in rs if ECD_FIRST <= r <= ECD_LAST]
    print("chain %s: %d residues %d..%d, %d in the ECD, %d gaps"
          % (c, len(rs), min(rs), max(rs), len(inside), len(gaps)))
    if gaps:
        fatal("chain %s has breaks at %s. CHARMM-GUI will either refuse the file or "
              "model the loop; decide which before running." % (c, gaps))
    if not (min(rs) <= ECD_FIRST and max(rs) >= ECD_LAST):
        fatal("chain %s covers %d..%d, the ECD asked for is %d..%d"
              % (c, min(rs), max(rs), ECD_FIRST, ECD_LAST))
    got = res.get((c, POSITION))
    if got != WT_RESNAME:
        fatal("position %d in chain %s is %s, not %s. Check the numbering: this "
              "structure should use UniProt P78348 numbering."
              % (POSITION, c, got, WT_RESNAME))

# Disulfides are measured, not assumed; a missed pair unfolds the domain.
sg = {}
for L in lines:
    if L[17:20].strip() == "CYS" and L[12:16].strip() == "SG":
        r = int(L[22:26])
        if ECD_FIRST <= r <= ECD_LAST:
            sg[(L[21], r)] = (float(L[30:38]), float(L[38:46]), float(L[46:54]))
ss, keys = [], sorted(sg)
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        if a[0] == b[0] and math.dist(sg[a], sg[b]) < 2.5:
            ss.append((a[0], a[1], b[1], round(math.dist(sg[a], sg[b]), 2)))
per_chain = collections.Counter(c for c, _, _, _ in ss)
print("\ndisulfides inside the ECD, per chain:", dict(per_chain))
for c, a, b, d in ss:
    if c == chains[0]:
        print("   CYS%d - CYS%d  %.2f A" % (a, b, d))
if any(per_chain.get(c, 0) != EXPECT_SS for c in chains):
    fatal("expected %d disulfides per chain inside the ECD, found %s. Do not hand "
          "CHARMM-GUI a list you have not checked." % (EXPECT_SS, dict(per_chain)))
free = sorted(r for (c, r) in sg if c == chains[0]
              and not any(r in (a, b) for cc, a, b, _ in ss if cc == chains[0]))
print("free cysteines in the ECD, per chain:", free)


def write_build(tag, new_resname):
    """One trimmed PDB with the substitution applied to every chain."""
    keep = None
    if new_resname != WT_RESNAME:
        if new_resname not in SIDECHAIN or WT_RESNAME not in SIDECHAIN:
            fatal("no side-chain definition for %s or %s; add it to SIDECHAIN above"
                  % (WT_RESNAME, new_resname))
        old, new = SIDECHAIN[WT_RESNAME], SIDECHAIN[new_resname]
        shared = 0
        for x, y in zip(old, new):
            if x != y:
                break
            shared += 1
        keep = set(BACKBONE) | set(old[:shared])
        print("  %-8s keeps %s, CHARMM-GUI rebuilds %s"
              % (tag, sorted(set(old[:shared])) or "backbone only", new[shared:]))
    out, n, nmut = [], 0, 0
    for L in lines:
        r = int(L[22:26])
        if not (ECD_FIRST <= r <= ECD_LAST):
            continue
        if keep is not None and r == POSITION:
            if L[12:16].strip() not in keep:
                continue
            L = L[:17] + "%-3s" % new_resname + L[20:]
            nmut += 1
        n += 1
        out.append(L)
    if keep is not None and nmut == 0:
        fatal("%s: no atoms were renamed at position %d" % (tag, POSITION))
    path = os.path.join(OUT_DIR, "%s_ECD_%s.pdb" % (PDB_ID, tag))
    with open(path, "w") as fh:
        fh.write("REMARK  ASIC1a pI-code MD input, %s\n" % tag)
        fh.write("REMARK  %s residues %d..%d of chains %s\n"
                 % (PDB_ID, ECD_FIRST, ECD_LAST, ",".join(chains)))
        if keep is not None:
            fh.write("REMARK  %s%d%s applied to every chain\n"
                     % (WT_RESNAME, POSITION, new_resname))
        fh.write("\n".join(out) + "\nEND\n")
    print("           %-24s %d atoms" % (os.path.basename(path), n))
    return path, n


print("\nbuilds:")
BUILDS = collections.OrderedDict()
for tag, rn in (("wt", WT_RESNAME), ("ladder", LADDER_TO), ("compare", COMPARE_TO)):
    BUILDS[tag] = write_build(tag, rn)

with open(os.path.join(OUT_DIR, "disulfides.txt"), "w") as fh:
    for c, a, b, d in ss:
        fh.write("%s CYS%d CYS%d %.2f\n" % (c, a, b, d))

# Part 2 reads this, so the two notebooks cannot disagree about which build is which.
spec = dict(pdb_id=PDB_ID, position=POSITION, ecd=[ECD_FIRST, ECD_LAST],
            chains=chains, disulfides_per_chain=EXPECT_SS,
            expect={"wt": WT_RESNAME, "ladder": LADDER_TO, "compare": COMPARE_TO},
            atoms={k: v[1] for k, v in BUILDS.items()})
json.dump(spec, open(os.path.join(OUT_DIR, "build_spec.json"), "w"), indent=2)

CHECKLIST = """CHARMM-GUI settings for the ASIC1a pI-code MD
=============================================
Input Generator > Solution Builder > Upload PDB. Run it three times, once per file.

  %s
  %s
  %s

Chains            keep all three, %s. The pocket is a two-subunit cavity.
Disulfides        %d per chain, %d in all. The pairs are in disulfides.txt beside these
                  files. Confirm every one is ticked: CHARMM-GUI does not always detect
                  them all, and a missed pair unfolds the domain.
Terminal groups   ACE at the N terminus and CT3 at the C terminus of every chain. The
                  termini here are an artificial cut through the middle of a protein,
                  so charged termini would put six formal charges where the real
                  protein has none.
Protonation       defaults, standard protonation at pH 7. The three builds must differ
                  in nothing but the substitution.
Water box         rectangular, 12 angstrom edge distance.
Ions              KCl 0.15 M, neutralising. The three builds carry different net charge
                  so the ion counts will differ; that is correct.
Force field       CHARMM36m.
Dynamics          GROMACS. Temperature 310 K.

Unpack each charmm-gui-*.tgz and put the gromacs/ folder on Drive as

  %s/wt/gromacs
  %s/ladder/gromacs
  %s/compare/gromacs

Then open the three part 2 notebooks, one per build, and press Runtime > Run all on
each. They are independent and can run at the same time.
""" % (os.path.basename(BUILDS["wt"][0]), os.path.basename(BUILDS["ladder"][0]),
       os.path.basename(BUILDS["compare"][0]), ",".join(chains), EXPECT_SS,
       EXPECT_SS * len(chains), PROJECT, PROJECT, PROJECT)

open(os.path.join(OUT_DIR, "CHARMM-GUI-checklist.txt"), "w").write(CHECKLIST)
print("\n" + "=" * 72)
print(CHECKLIST)
print("everything written to", OUT_DIR)

## Done

The three PDBs, `disulfides.txt`, `build_spec.json` and `CHARMM-GUI-checklist.txt` are
in `inputs/` under your project folder on Drive. The checklist printed above is the same
file, so you can close this notebook and work from Drive.

Part 2 reads `build_spec.json` and refuses any folder whose residue at the mutated
position does not match what this notebook wrote, so uploading the three builds in the
wrong order is caught before anything runs.